# Amazon Reviews Sentiment Analysis

## 1. Data Loading and Cleaning

In [ ]:
import re
import string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

sns.set_style('whitegrid')

In [ ]:
df = pd.read_csv('amazonreviews.tsv', sep='\t')
df.shape

In [ ]:
df.head()

In [ ]:
df['label'].value_counts()

In [ ]:
df.isnull().sum()

In [ ]:
df.dropna(subset=['review'], inplace=True)
df.drop_duplicates(subset='review', keep='first', inplace=True)
df.reset_index(drop=True, inplace=True)
df.shape

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    words = [w for w in text.split() if w not in ENGLISH_STOP_WORDS and len(w) > 2]
    return ' '.join(words)

df['clean_review'] = df['review'].apply(clean_text)
df[['review', 'clean_review']].head()

In [ ]:
df['review_length'] = df['clean_review'].apply(lambda x: len(x.split()))
df['review_length'].describe()

## 2. Exploratory Analysis

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x='label', data=df, order=['pos', 'neg'])
plt.title('Sentiment Distribution')
plt.xlabel('Sentiment')
plt.ylabel('Count')
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df['review_length'], bins=50, kde=True)
plt.title('Distribution of Review Lengths (words)')
plt.xlabel('Number of Words')
plt.show()

In [ ]:
pos_text = ' '.join(df[df['label'] == 'pos']['clean_review'])
neg_text = ' '.join(df[df['label'] == 'neg']['clean_review'])

pos_wc = WordCloud(width=800, height=400, background_color='white', max_words=100).generate(pos_text)
neg_wc = WordCloud(width=800, height=400, background_color='white', max_words=100).generate(neg_text)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(pos_wc, interpolation='bilinear')
axes[0].axis('off')
axes[0].set_title('Positive Reviews Word Cloud')
axes[1].imshow(neg_wc, interpolation='bilinear')
axes[1].axis('off')
axes[1].set_title('Negative Reviews Word Cloud')
plt.show()

In [ ]:
def get_top_words(text, n=20):
    words = text.split()
    counts = Counter(words)
    return counts.most_common(n)

pos_top = get_top_words(pos_text, 20)
neg_top = get_top_words(neg_text, 20)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

pos_words, pos_counts = zip(*pos_top)
axes[0].barh(pos_words[::-1], pos_counts[::-1], color='green')
axes[0].set_title('Top 20 Words in Positive Reviews')

neg_words, neg_counts = zip(*neg_top)
axes[1].barh(neg_words[::-1], neg_counts[::-1], color='red')
axes[1].set_title('Top 20 Words in Negative Reviews')

plt.tight_layout()
plt.show()

## 3. Feature Extraction

In [ ]:
X = df['clean_review']
y = df['label'].map({'pos': 1, 'neg': 0})

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train.shape, X_test.shape

In [ ]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)
X_train_tfidf.shape, X_test_tfidf.shape

## 4. Model Development

In [ ]:
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_tfidf, y_train)
log_reg_preds = log_reg.predict(X_test_tfidf)

In [ ]:
svm_model = LinearSVC(random_state=42)
svm_model.fit(X_train_tfidf, y_train)
svm_preds = svm_model.predict(X_test_tfidf)

## 5. Validation

In [ ]:
log_reg_cv = cross_val_score(LogisticRegression(max_iter=1000, random_state=42), X_train_tfidf, y_train, cv=5, scoring='f1')
svm_cv = cross_val_score(LinearSVC(random_state=42), X_train_tfidf, y_train, cv=5, scoring='f1')

print('Logistic Regression CV F1 scores:', log_reg_cv)
print('Logistic Regression CV F1 mean:', log_reg_cv.mean())
print('SVM CV F1 scores:', svm_cv)
print('SVM CV F1 mean:', svm_cv.mean())

In [ ]:
log_reg_acc = accuracy_score(y_test, log_reg_preds)
log_reg_f1 = f1_score(y_test, log_reg_preds)

svm_acc = accuracy_score(y_test, svm_preds)
svm_f1 = f1_score(y_test, svm_preds)

results_df = pd.DataFrame({
    'Model': ['Logistic Regression', 'SVM (Linear)'],
    'Test Accuracy': [log_reg_acc, svm_acc],
    'Test F1 Score': [log_reg_f1, svm_f1],
    'CV F1 Mean': [log_reg_cv.mean(), svm_cv.mean()]
})
results_df

In [ ]:
print('Logistic Regression Classification Report')
print(classification_report(y_test, log_reg_preds, target_names=['neg', 'pos']))

print('SVM Classification Report')
print(classification_report(y_test, svm_preds, target_names=['neg', 'pos']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm_log = confusion_matrix(y_test, log_reg_preds)
sns.heatmap(cm_log, annot=True, fmt='d', cmap='Blues', xticklabels=['neg', 'pos'], yticklabels=['neg', 'pos'], ax=axes[0])
axes[0].set_title('Logistic Regression Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

cm_svm = confusion_matrix(y_test, svm_preds)
sns.heatmap(cm_svm, annot=True, fmt='d', cmap='Greens', xticklabels=['neg', 'pos'], yticklabels=['neg', 'pos'], ax=axes[1])
axes[1].set_title('SVM Confusion Matrix')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()

In [ ]:
feature_names = np.array(tfidf.get_feature_names_out())
coefs = log_reg.coef_[0]

top_pos_idx = np.argsort(coefs)[-15:]
top_neg_idx = np.argsort(coefs)[:15]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].barh(feature_names[top_pos_idx], coefs[top_pos_idx], color='green')
axes[0].set_title('Top Positive-Sentiment Features (Logistic Regression)')

axes[1].barh(feature_names[top_neg_idx], coefs[top_neg_idx], color='red')
axes[1].set_title('Top Negative-Sentiment Features (Logistic Regression)')

plt.tight_layout()
plt.show()

## 6. Discussion and Conclusion

Both Logistic Regression and Linear SVM were trained on TF-IDF features (unigrams and bigrams, top 5000 features) extracted from cleaned review text. Cross-validation on the training set and evaluation on a held-out test set were used to compare the two models on accuracy and F1-score.

The word clouds and top-word frequency plots show that positive reviews are dominated by strongly favorable adjectives, while negative reviews are dominated by complaint-oriented and disappointment-related vocabulary, confirming that the text content carries clear sentiment signal beyond the star rating alone.

The confusion matrices show where each model makes errors — reviews with mixed sentiment (e.g. a mostly positive review with a minor complaint, or sarcasm) are the most common source of misclassification for both models, which is expected for models built purely on bag-of-words / TF-IDF representations without deeper contextual understanding.

Based on the test accuracy and F1-score comparison table, the better-performing model can be selected for deployment as the automated sentiment classifier. For further improvement, incorporating contextual embeddings (e.g. Word2Vec or BERT) instead of TF-IDF, handling negation explicitly, and using ensemble methods could improve performance further, especially on reviews with mixed or sarcastic sentiment.